# Tritium Sampling with a Bubbler

A bubbler collects tritium from a gas stream for measurement and inventory tracking. The gas bubbles through a series of liquid filled vials. Soluble tritium (HTO) is retained in the vials, insoluble tritium (HT, T$_2$) is first converted to a soluble form by a catalyst.

`Bubbler4` has two chains of two vials each. The soluble tritium $u_{sol}$ passes vials 1 and 2. The insoluble tritium $u_{insol}$ is converted with efficiency $\alpha$ and passes vials 3 and 4 together with the soluble tritium that was not retained in the first chain. Each vial retains the fraction $\eta$ of the tritium that reaches it:

$$\begin{align}
\dot{x}_1 &= \eta \, u_{sol} \\
\dot{x}_2 &= \eta \, (1 - \eta) \, u_{sol} \\
\dot{x}_3 &= \eta \, \left[ \alpha \, u_{insol} + (1 - \eta)^2 \, u_{sol} \right] \\
\dot{x}_4 &= \eta \, (1 - \eta) \left[ \alpha \, u_{insol} + (1 - \eta)^2 \, u_{sol} \right]
\end{align}$$

The tritium that is not retained leaves the bubbler as `sample_out`. Vials are replaced at given times, which resets their inventory to zero.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("../pathsim_docs.mplstyle")

from pathsim import Connection, Simulation
from pathsim.blocks import Constant, Integrator, Scope

from pathsim_fusion import Bubbler4

## Sampling with Vial Replacement

A constant stream with soluble and insoluble tritium is sampled over 30 days, all vials are replaced every 10 days. An integrator accumulates the tritium that leaves the bubbler. The values are illustrative and in arbitrary units.

In [ ]:
# tritium flow rates of the sampled stream
sol = Constant(1.0)
insol = Constant(0.5)

bubbler = Bubbler4(
    conversion_efficiency=0.95,
    vial_efficiency=0.9,
    replacement_times=[10.0, 20.0],
)

# tritium that is not retained
lost = Integrator()

sco = Scope(labels=["vial 1", "vial 2", "vial 3", "vial 4", "not retained"])

sim = Simulation(
    [sol, insol, bubbler, lost, sco],
    [
        Connection(sol, bubbler["sample_in_soluble"]),
        Connection(insol, bubbler["sample_in_insoluble"]),
        Connection(bubbler["vial1"], sco[0]),
        Connection(bubbler["vial2"], sco[1]),
        Connection(bubbler["vial3"], sco[2]),
        Connection(bubbler["vial4"], sco[3]),
        Connection(bubbler["sample_out"], lost),
        Connection(lost, sco[4]),
    ],
    dt=0.05,
    log=True,
)

sim.run(30)

In [ ]:
sco.plot(lw=1.5)
plt.xlabel("time [days]")
plt.ylabel("tritium inventory")
plt.title("Vial inventories with replacement every 10 days")
plt.show()

Vial 1 collects most of the soluble tritium, vial 3 most of the converted insoluble tritium. The inventories drop to zero at each replacement. The fraction of tritium that leaves the bubbler follows from the efficiencies:

In [ ]:
eta, alpha = 0.9, 0.95
u_sol, u_insol = 1.0, 0.5

rate_out = (1 - alpha) * u_insol + (1 - eta) ** 2 * (
    alpha * u_insol + (1 - eta) ** 2 * u_sol
)
print(f"not retained: {rate_out / (u_sol + u_insol):.2%} of the sampled tritium")